In [1]:
"""PET NER dataset download, loading, and pool/test splitting."""

import urllib.request
from pathlib import Path

from config import (
    FEW_SHOT_SPLIT_SEED,
    N_FEW_SHOT_EXAMPLES,
    NER_DATASET_URL,
    NER_TAGS,
    RAW_DATASET_PATH,
    SEED,
    TEST_SIZE,
)
from datasets import ClassLabel, Dataset, Features, Sequence, Value, load_dataset


def download_pet_ner(raw_data_path: Path = RAW_DATASET_PATH, force: bool = False) -> Path:
    """Download the PET entities jsonl into data/raw/."""
    if raw_data_path.exists() and not force:
        return raw_data_path
    raw_data_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(NER_DATASET_URL, raw_data_path)
    return raw_data_path


def load_pet_ner(path: Path = RAW_DATASET_PATH) -> Dataset:
    """Load the PET entities dataset (417 sentence-level examples) from data/raw/."""
    if not path.exists():
        raise FileNotFoundError(f"{path} not found — run `uv run uq-pet download-data` first.")
    features = Features(
        {
            "document name": Value("string"),
            "sentence-ID": Value("int8"),
            "tokens": Sequence(Value("string")),
            "ner-tags": Sequence(ClassLabel(names=NER_TAGS)),
        }
    )
    return load_dataset("json", data_files={"full": str(path)}, features=features)["full"]


def split_dataset(
    seed: int = SEED,
    test_size: float = TEST_SIZE,
    n_few_shot: int = N_FEW_SHOT_EXAMPLES,
    few_shot_seed: int = FEW_SHOT_SPLIT_SEED,
) -> tuple[Dataset, Dataset, Dataset]:
    """Split PET into few-shot examples, experiment pool, and held-out test.

    Returns (few_shot, pool, test). The pool excludes the few-shot examples.
    """
    outer = load_pet_ner().train_test_split(test_size=test_size, seed=seed)
    inner = outer["train"].train_test_split(train_size=n_few_shot, shuffle=True, seed=few_shot_seed)
    return inner["train"], inner["test"], outer["test"]


def tag_ids_to_labels(tag_ids: list) -> list[str]:
    return [NER_TAGS[tid] for tid in tag_ids]

In [2]:
print(NER_DATASET_URL, "\n", NER_TAGS, "\n", RAW_DATASET_PATH, "\n", SEED, "\n", TEST_SIZE)

https://raw.githubusercontent.com/patriziobellan86/PETv1.1/master/PETv1.1-entities.jsonl 
 ['O', 'B-Actor', 'I-Actor', 'B-Activity', 'I-Activity', 'B-Activity Data', 'I-Activity Data', 'B-Further Specification', 'I-Further Specification', 'B-XOR Gateway', 'I-XOR Gateway', 'B-Condition Specification', 'I-Condition Specification', 'B-AND Gateway', 'I-AND Gateway'] 
 /Users/mac/Developer/VScode/uq-pet/data/raw/PETv1.1-entities.jsonl 
 3407 
 0.2


In [3]:
download_pet_ner(force=True)

PosixPath('/Users/mac/Developer/VScode/uq-pet/data/raw/PETv1.1-entities.jsonl')

## Prompt creation

In [ ]:
from string import Template
from textwrap import dedent

TAG_LEGEND = "\n".join(f"  {i} = {tag}" for i, tag in enumerate(NER_TAGS))

SYSTEM_TEMPLATE = Template(
    dedent("""\
    You are a strict Named Entity Recognition (NER) system for Process Extraction.
    Assign exactly one tag ID to each token in the sentence.

    ENTITY DEFINITIONS:
    - Actor: The person, system, or role performing the action.
    - Activity: The task or action being executed.
    - Activity Data: The object, document, or data manipulated by the activity.
    - Further Specification: Additional context, tools, or locations (e.g., 'via email').
    - XOR Gateway: Words indicating an exclusive branching point (e.g., 'If', 'otherwise').
    - Condition Specification: The condition required to take a branch (e.g., 'the claim is valid').
    - AND Gateway: Words indicating parallel execution (e.g., 'in parallel').
    - O: Tokens outside of any process entity.

    DATASET RULES:
    - Determiners ('The', 'a', 'an') MUST be included in the entity if they precede it.
    - Multi-word entities must start with 'B-' (Beginning) and continue with 'I-' (Inside).
    - Single-word entities get the 'B-' tag.

    TAG IDS:
    $tag_legend

    === EXAMPLES ===
    $examples
    === END OF EXAMPLES ===
    """)
)

USER_TEMPLATE = Template(
    dedent("""\
    Tokens to tag:
    $tokens

    Output MUST be an array of integers with exactly $n_tokens elements, one per
    token, in order. Output nothing except the array.
    """)
)


def build_system_prompt(few_shot: Dataset) -> str:
    """Build the constant prompt prefix — identical for every sentence, so it caches."""
    examples = "\n\n".join(f"{ex['tokens']}\n{ex['ner-tags']}" for ex in few_shot)
    return SYSTEM_TEMPLATE.substitute(tag_legend=TAG_LEGEND, examples=examples)


def build_user_prompt(tokens: list[str]) -> str:
    """Build the user prompt for one sentence of PET tokens."""
    return USER_TEMPLATE.substitute(tokens=tokens, n_tokens=len(tokens))

few_shot, pool, test = split_dataset()

system_prompt = build_system_prompt(few_shot)
pool_messages = [build_user_prompt(ex["tokens"]) for ex in pool]
test_messages = [build_user_prompt(ex["tokens"]) for ex in test]

Generating full split: 0 examples [00:00, ? examples/s]

## UQ using LLMs for NER

In [12]:
print(system_prompt)

You are a strict Named Entity Recognition (NER) system for Process Extraction.
Assign exactly one tag ID to each token in the sentence.

ENTITY DEFINITIONS:
- Actor: The person, system, or role performing the action.
- Activity: The task or action being executed.
- Activity Data: The object, document, or data manipulated by the activity.
- Further Specification: Additional context, tools, or locations (e.g., 'via email').
- XOR Gateway: Words indicating an exclusive branching point (e.g., 'If', 'otherwise').
- Condition Specification: The condition required to take a branch (e.g., 'the claim is valid').
- AND Gateway: Words indicating parallel execution (e.g., 'in parallel').
- O: Tokens outside of any process entity.

DATASET RULES:
- Determiners ('The', 'a', 'an') MUST be included in the entity if they precede it.
- Multi-word entities must start with 'B-' (Beginning) and continue with 'I-' (Inside).
- Single-word entities get the 'B-' tag.

TAG IDS:
  0 = O
  1 = B-Actor
  2 = I-Act

In [5]:
pool_messages[0]

"Tokens to tag:\n['In', 'the', 'meantime', ',', 'the', 'engineering', 'department', 'prepares', 'everything', 'for', 'the', 'assembling', 'of', 'the', 'ordered', 'bicycle', '.']\n\nOutput MUST be a JSON array of integers with exactly 17 elements, one per\ntoken, in order. Output nothing except the array.\n"

In [11]:
pool[0]

{'document name': 'doc-1.1',
 'sentence-ID': 9,
 'tokens': ['In',
  'the',
  'meantime',
  ',',
  'the',
  'engineering',
  'department',
  'prepares',
  'everything',
  'for',
  'the',
  'assembling',
  'of',
  'the',
  'ordered',
  'bicycle',
  '.'],
 'ner-tags': [13, 14, 14, 0, 1, 2, 2, 3, 5, 0, 0, 0, 0, 0, 0, 0, 0]}

### LLM part


In [6]:
from openai import OpenAI
import os

# Initialize client with private endpoint URL and API key
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.getenv("NHR_FAU_API_KEY"),
    base_url="https://hub.nhr.fau.de/api/llmgw/v1"
)

# Create a chat completion request
response = client.chat.completions.create(
    model="RedHatAI/gemma-4-31B-it-FP8-block", # Replace with a model name available to you!
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": pool_messages[0]}
    ],
    temperature=0.7, # Optional parameter
    logprobs=True,
    top_logprobs=20
)

# Print the response
print(response.choices[0].message.content)

[0, 0, 0, 0, 1, 2, 2, 3, 5, 0, 5, 6, 6, 6, 6, 6, 0]


In [9]:
import math
import pandas as pd


def tag_rows(response, k: int = 5) -> pd.DataFrame:
    content = response.choices[0].logprobs.content
    rows, buf = [], []

    def flush():
        if not buf:
            return
        first = content[buf[0]]
        row = {
            "tag_pos": len(rows),
            "tag_id": int("".join(content[i].token.strip() for i in buf)),
            "n_tokens": len(buf),
            "p": math.exp(first.logprob),
            "mass_in_top": sum(math.exp(a.logprob) for a in first.top_logprobs),
            "entropy": -sum(math.exp(a.logprob) * a.logprob for a in first.top_logprobs),
        }
        for j, a in enumerate(first.top_logprobs[:k]):
            row[f"alt{j}"] = a.token.strip()
            row[f"alt{j}_p"] = math.exp(a.logprob)
        rows.append(row)
        buf.clear()

    for i, tok in enumerate(content):
        if tok.token.strip().isdigit():
            buf.append(i)
        else:
            flush()
    flush()
    return pd.DataFrame(rows)


def style_tag_rows(df: pd.DataFrame, k: int = 5):
    alt_p = [f"alt{j}_p" for j in range(k)]
    return (
        df.style
        .background_gradient(cmap="Reds", subset=["entropy"])
        .background_gradient(cmap="RdYlGn", subset=["p", "mass_in_top"], vmin=0, vmax=1)
        .background_gradient(cmap="Blues", subset=alt_p, vmin=0, vmax=1)
        .format({"p": "{:.4f}", "mass_in_top": "{:.6f}", "entropy": "{:.4f}",
                 **{c: "{:.4f}" for c in alt_p}})
    )

df = tag_rows(response)
tokens = pool[0]["tokens"]

df.insert(1, "token", tokens)
df.insert(3, "tag", [NER_TAGS[t] for t in df["tag_id"]])

style_tag_rows(df)


,tag_pos,token,tag_id,tag,n_tokens,p,mass_in_top,entropy,alt0,alt0_p,alt1,alt1_p,alt2,alt2_p,alt3,alt3_p,alt4,alt4_p
0,0,In,0,O,1,1.0000,1.000000,0.0000,0,1.0000,7,0.0000,,0.0000,9,0.0000,8,0.0000
1,1,the,0,O,1,1.0000,1.000000,0.0005,0,1.0000,1,0.0000,7,0.0000,8,0.0000,5,0.0000
2,2,meantime,0,O,1,1.0000,1.000000,0.0001,0,1.0000,7,0.0000,8,0.0000,1,0.0000,2,0.0000
3,3,",",0,O,1,1.0000,1.000000,0.0000,0,1.0000,1,0.0000,2,0.0000,4,0.0000,_,0.0000
4,4,the,1,B-Actor,1,1.0000,1.000000,0.0000,1,1.0000,5,0.0000,2,0.0000,3,0.0000,8,0.0000
5,5,engineering,2,I-Actor,1,1.0000,1.000000,0.0000,2,1.0000,1,0.0000,3,0.0000,0,0.0000,//,0.0000
6,6,department,2,I-Actor,1,1.0000,1.000000,0.0001,2,1.0000,3,0.0000,1,0.0000,0,0.0000,8,0.0000
7,7,prepares,3,B-Activity,1,1.0000,1.000000,0.0000,3,1.0000,0,0.0000,4,0.0000,_,0.0000,//,0.0000
8,8,everything,5,B-Activity Data,1,0.9997,1.000000,0.0034,5,0.9997,0,0.0003,4,0.0000,3,0.0000,6,0.0000
9,9,for,0,O,1,0.8665,0.999999,0.5286,0,0.8665,7,0.0806,6,0.0296,8,0.0204,4,0.0028


### UQ part
